# Dataset Compare: FaceGuard Cross-Dataset Evaluation

This notebook is based on `FaceGuard_Simplified_MultiDataset.ipynb` and implements the updated cross-dataset plan for exactly three datasets: LFW, CelebA, and VGGFace2. LFW is the reference dataset for dataset-shift degradation.

Main configuration: image size 224 x 224, InceptionResnetV1 pretrained on VGGFace2, epsilon 12/255 for the main comparison, 35 optimization steps, step-size ratio 0.125, central facial mask, and seeds [0, 1, 2].


## 0. Optional Google Drive Mount
Run this in Colab when datasets or outputs live in Google Drive.


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('[drive] skipped:', exc)


Mounted at /content/drive


In [5]:
%%bash
SRC_BASE="/content/drive/My Drive/FaceGuard/Datasets"
DEST_BASE="datasets"

mkdir -p "$DEST_BASE/celeba/img_align"
if [ -z "$(ls -A "$DEST_BASE/celeba/img_align" 2>/dev/null)" ]; then
  unzip -q "$SRC_BASE/celeba/img_align_celeba.zip" -d "$DEST_BASE/celeba/img_align"
  echo "CelebA extracted"
else
  echo "CelebA already extracted, skipping"
fi

mkdir -p "$DEST_BASE/vgg-face2/images"
# for tar in "$SRC_BASE/vgg-face2/data/vggface2_train.tar.gz" "$SRC_BASE/vgg-face2/data/vggface2_test.tar.gz"; do
#   name=$(basename "$tar" .tar.gz)
#   if [ ! -d "$DEST_BASE/vgg-face2/images/$name" ]; then
#     tar -xzf "$tar" -C "$DEST_BASE/vgg-face2/images"
#     echo "$name extracted"
#   else
#     echo "$name already exists, skipping"
#   fi
# done

name="vggface2_test"

if [ ! -d "$DEST_BASE/vgg-face2/images/$name" ]; then
  tar -xzf "$SRC_BASE/vgg-face2/data/${name}.tar.gz" -C "$DEST_BASE/vgg-face2/images"
  echo "$name extracted"
else
  echo "$name already exists, skipping"
fi


CelebA extracted
vggface2_test extracted


## 1. Install Dependencies
After this cell finishes in Colab, restart the runtime before continuing if package versions changed.


In [2]:
!apt-get update -y && apt-get install -y curl wget
%pip install --upgrade --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2" "scikit-image==0.24.0" "pandas==2.2.2" "matplotlib==3.9.2" "opencv-python-headless==4.10.0.84" "facenet-pytorch==2.6.0" "lpips==0.1.4" "tqdm==4.66.5"
print('[setup] Restart the runtime/kernel after this install cell before running imports.')


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,078 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,52

In [3]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126


Looking in indexes: https://download.pytorch.org/whl/cu126


## 2. Configuration
Edit dataset paths before running. Use `N_IMAGES = 500` for the recommended experiment or `N_IMAGES = 100` when GPU time/storage is limited.


In [3]:
import os, time, random, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

SEEDS = [0, 1, 2]
IMAGE_SIZE = 224
N_IMAGES = 500
RUN_OPTIONAL_EPSILON_SWEEP = False

MAIN_EPSILON = 12 / 255
EPSILON_SWEEP = [4 / 255, 8 / 255, 12 / 255, 16 / 255]
EPSILON_LIST = EPSILON_SWEEP if RUN_OPTIONAL_EPSILON_SWEEP else [MAIN_EPSILON]
STEPS = 35
STEP_RATIO = 0.125
USE_EOT = True
FACEGUARD_BATCH_SIZE = 8
BASELINE_BATCH_SIZE = 8

SUCCESS_COSINE_THRESHOLD = 0.0
RELAXED_SUCCESS_COSINE_THRESHOLD = 0.3
SUCCESS_PSNR_THRESHOLD = 30.0

DATASET_PATHS = {
    'LFW': None,
    'CelebA': '/content/datasets/celeba/img_align/img_align_celeba',
    'VGGFace2': '/content/datasets/vgg-face2/images/test',
}
CELEBA_IDENTITY_FILE = '/content/drive/My Drive/FaceGuard/Datasets/celeba/anno/identity_CelebA.txt'

RUNTIME_ROOT = '/content' if os.path.isdir('/content') else os.path.join(os.getcwd(), 'faceguard_runtime')
DRIVE_ROOT = '/content/drive/My Drive/FaceGuard' if os.path.isdir('/content/drive/My Drive') else None
SAVE_ROOT = DRIVE_ROOT if DRIVE_ROOT else RUNTIME_ROOT
EXPERIMENT_ROOT = os.path.join(SAVE_ROOT, 'experiments')
CONFIG_DIR = os.path.join(EXPERIMENT_ROOT, 'configs')
RESULTS_DIR = os.path.join(EXPERIMENT_ROOT, 'results')
TABLE_DIR = os.path.join(EXPERIMENT_ROOT, 'tables')
FIGURE_DIR = os.path.join(EXPERIMENT_ROOT, 'figures')
for d in [CONFIG_DIR, RESULTS_DIR, TABLE_DIR, FIGURE_DIR]:
    os.makedirs(d, exist_ok=True)

CONFIG = {
    'datasets': ['LFW', 'CelebA', 'VGGFace2'],
    'reference_dataset': 'LFW',
    'image_size': IMAGE_SIZE,
    'n_images_per_dataset': N_IMAGES,
    'identity_extractor': 'InceptionResnetV1 pretrained on VGGFace2',
    'main_epsilon': MAIN_EPSILON,
    'epsilon_list': EPSILON_LIST,
    'steps': STEPS,
    'step_ratio': STEP_RATIO,
    'mask': 'central facial region mask',
    'seeds': SEEDS,
}
config_path = os.path.join(CONFIG_DIR, 'cross_dataset_lfw_celeba_vggface2.yaml')
with open(config_path, 'w', encoding='utf-8') as f:
    for key, value in CONFIG.items():
        f.write(f'{key}: {json.dumps(value)}\n')
print('[config] saved', config_path)
print('[config] output root =', EXPERIMENT_ROOT)


[config] saved /content/drive/My Drive/FaceGuard/experiments/configs/cross_dataset_lfw_celeba_vggface2.yaml
[config] output root = /content/drive/My Drive/FaceGuard/experiments


## 3. Imports, Device, and Reproducibility


In [4]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from sklearn.datasets import fetch_lfw_people
from facenet_pytorch import InceptionResnetV1
import lpips

versions = {
    'numpy': np.__version__,
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
}
print('[versions]', versions)
if not np.__version__.startswith('1.26.'):
    print('[warning] The source notebook expects NumPy 1.26.x. Restart the runtime after installing dependencies if needed.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[env] device =', device)
if torch.cuda.is_available():
    print('[env] gpu =', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEEDS[0])


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[versions] {'numpy': '1.26.4', 'torch': '2.2.2+cu121', 'cuda_available': True}
[env] device = cuda
[env] gpu = Tesla T4


## 4. Image, Metric, and Export Helpers
All images are RGB, resized to 224 x 224, and represented as tensors normalized to [0, 1].


In [5]:
def ensure_uint8_rgb(img):
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[-1] == 4:
        img = img[..., :3]
    if img.dtype != np.uint8:
        if img.max() <= 1.0:
            img = img * 255.0
        img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def resize_rgb(img, size=IMAGE_SIZE):
    return np.asarray(Image.fromarray(ensure_uint8_rgb(img)).resize((size, size), Image.BICUBIC))

def image_to_tensor(img_rgb):
    arr = resize_rgb(img_rgb).astype(np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)

def tensor_to_rgb_uint8(x):
    arr = x.detach().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    return (arr * 255.0 + 0.5).astype(np.uint8)

def tensor_batch_to_rgb_uint8(x):
    arr = x.detach().clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()
    return (arr * 255.0 + 0.5).astype(np.uint8)

def rgb_to_tensor_device(img_rgb):
    return image_to_tensor(img_rgb).unsqueeze(0).to(device, non_blocking=True)

def rgb_to_bgr(img):
    return cv2.cvtColor(ensure_uint8_rgb(img), cv2.COLOR_RGB2BGR)

def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def psnr_np_rgb(a, b):
    a = ensure_uint8_rgb(a).astype(np.float32)
    b = ensure_uint8_rgb(b).astype(np.float32)
    mse = np.mean((a - b) ** 2)
    return float('inf') if mse == 0 else float(20 * np.log10(255.0 / np.sqrt(mse)))

def ssim_np_rgb(a, b):
    return float(ssim(ensure_uint8_rgb(a), ensure_uint8_rgb(b), channel_axis=2, data_range=255))

def make_face_mask(batch_size=1, h=IMAGE_SIZE, w=IMAGE_SIZE):
    mask = np.zeros((h, w), dtype=np.float32)
    cv2.ellipse(mask, (w // 2, int(h * 0.52)), (int(w * 0.34), int(h * 0.42)), 0, 0, 360, 1, -1)
    return torch.from_numpy(mask).view(1, 1, h, w).repeat(batch_size, 1, 1, 1).to(device)

def save_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False)
    print('[saved csv]', path)
    return path

def save_latex(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_latex(path, index=False, escape=False, float_format='%.4f')
    print('[saved table]', path)
    return path

def chunked(seq, size):
    for start in range(0, len(seq), size):
        yield start, seq[start:start + size]


## 5. Unified Dataset Loaders
Each loader returns rows with `image_tensor`, `image_id`, `dataset_name`, and `identity_id` when available.


In [6]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXTS])

def load_image_file(path):
    return resize_rgb(Image.open(path).convert('RGB'), IMAGE_SIZE)

def pack_sample(img_rgb, image_id, dataset_name, identity_id=None):
    return {
        'image_tensor': image_to_tensor(img_rgb),
        'image_id': str(image_id),
        'dataset_name': dataset_name,
        'identity_id': None if identity_id is None else str(identity_id),
    }

def deterministic_take(items, n, seed):
    items = list(items)
    rng = random.Random(seed)
    rng.shuffle(items)
    return items[:min(n, len(items))]

def load_lfw(n=N_IMAGES, seed=0):
    lfw = fetch_lfw_people(color=True, resize=1.0, funneled=True, min_faces_per_person=1)
    indices = deterministic_take(range(len(lfw.images)), n, seed)
    samples = []
    for idx in indices:
        identity = lfw.target_names[int(lfw.target[idx])]
        samples.append(pack_sample(lfw.images[idx], f'lfw_{idx:05d}', 'LFW', identity))
    return samples

def load_celeba(n=N_IMAGES, seed=0, folder=None, identity_file=None):
    folder = folder or DATASET_PATHS['CelebA']
    if folder is None or not os.path.exists(folder):
        raise FileNotFoundError(f'CelebA folder not found: {folder}')
    identity_lookup = {}
    if identity_file and os.path.exists(identity_file):
        id_df = pd.read_csv(identity_file, sep=r'\s+', names=['file', 'identity'])
        identity_lookup = dict(zip(id_df['file'].astype(str), id_df['identity'].astype(str)))
    files = deterministic_take(list_images(folder), n, seed)
    samples = []
    for path in files:
        identity = identity_lookup.get(path.name)
        samples.append(pack_sample(load_image_file(path), path.stem, 'CelebA', identity))
    return samples

def load_vggface2(n=N_IMAGES, seed=0, folder=None):
    folder = folder or DATASET_PATHS['VGGFace2']
    if folder is None or not os.path.exists(folder):
        raise FileNotFoundError(f'VGGFace2 folder not found: {folder}')
    files = deterministic_take(list_images(folder), n, seed)
    samples = []
    for path in files:
        identity = path.parent.name if path.parent != Path(folder) else None
        samples.append(pack_sample(load_image_file(path), path.stem, 'VGGFace2', identity))
    return samples

def load_all_datasets(n=N_IMAGES, seed=0):
    loaders = {
        'LFW': lambda: load_lfw(n, seed),
        'CelebA': lambda: load_celeba(n, seed, DATASET_PATHS['CelebA'], CELEBA_IDENTITY_FILE),
        'VGGFace2': lambda: load_vggface2(n, seed, DATASET_PATHS['VGGFace2']),
    }
    datasets = {}
    coverage = []
    for name, loader in loaders.items():
        try:
            samples = loader()
            datasets[name] = samples
            coverage.append({'dataset': name, 'status': 'loaded', 'N': len(samples)})
        except Exception as exc:
            coverage.append({'dataset': name, 'status': 'error: ' + repr(exc), 'N': 0})
    coverage_df = pd.DataFrame(coverage)
    display(coverage_df)
    missing = coverage_df[coverage_df['N'].eq(0)]
    if len(missing):
        raise RuntimeError('One or more required datasets did not load. Fix paths above and rerun. ' + missing.to_string(index=False))
    min_n = min(len(v) for v in datasets.values())
    if min_n < n:
        print(f'[dataset] using N={min_n} for all datasets because at least one dataset has fewer than requested {n} images.')
        datasets = {name: samples[:min_n] for name, samples in datasets.items()}
    return datasets, coverage_df

datasets, dataset_coverage = load_all_datasets(N_IMAGES, SEEDS[0])


,dataset,status,N
0,LFW,loaded,500
1,CelebA,loaded,500
2,VGGFace2,loaded,500


## 6. Identity Model, LPIPS, and FaceGuard


In [7]:
identity_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
for param in identity_model.parameters():
    param.requires_grad_(False)

lpips_model = lpips.LPIPS(net='alex').eval().to(device)
for param in lpips_model.parameters():
    param.requires_grad_(False)

def facenet_preprocess(x):
    return (x - 0.5) / 0.5

def identity_embedding_grad(x):
    return F.normalize(identity_model(facenet_preprocess(x)), dim=1)

@torch.no_grad()
def identity_embedding_no_grad(x):
    return F.normalize(identity_model(facenet_preprocess(x)), dim=1)

@torch.no_grad()
def cosine_identity_batch(x, y):
    return F.cosine_similarity(identity_embedding_no_grad(x), identity_embedding_no_grad(y)).detach().cpu().numpy()

@torch.no_grad()
def lpips_batch(x, y):
    x_lp = x * 2 - 1
    y_lp = y * 2 - 1
    return lpips_model(x_lp, y_lp).view(-1).detach().cpu().numpy()

def differentiable_resize(x, scale=0.80):
    h, w = x.shape[-2:]
    small = F.interpolate(x, size=(int(h * scale), int(w * scale)), mode='bilinear', align_corners=False)
    return F.interpolate(small, size=(h, w), mode='bilinear', align_corners=False)

def differentiable_blur(x, k=5, sigma=1.0):
    coords = torch.arange(k, device=x.device) - k // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel = (g[:, None] @ g[None, :]).view(1, 1, k, k).repeat(x.shape[1], 1, 1, 1)
    return F.conv2d(x, kernel, padding=k // 2, groups=x.shape[1])

def faceguard_loss(adv, original_embedding, use_eot=True, eot_weight=0.5):
    loss = F.cosine_similarity(identity_embedding_grad(adv), original_embedding).mean()
    if use_eot:
        loss = loss + eot_weight * F.cosine_similarity(identity_embedding_grad(differentiable_resize(adv)), original_embedding).mean()
        loss = loss + eot_weight * F.cosine_similarity(identity_embedding_grad(differentiable_blur(adv)), original_embedding).mean()
    return loss

def project_linf(adv, clean, eps):
    return torch.max(torch.min(adv, clean + eps), clean - eps).clamp(0, 1)

def protect_faceguard(x, eps=MAIN_EPSILON, alpha_ratio=STEP_RATIO, steps=STEPS, mask=None, use_eot=USE_EOT):
    if mask is None:
        mask = torch.ones_like(x[:, :1])
    original_embedding = identity_embedding_no_grad(x).detach()
    adv = (x + torch.empty_like(x).uniform_(-eps, eps) * mask).clamp(0, 1).detach()
    alpha = eps * alpha_ratio
    for _ in range(steps):
        adv.requires_grad_(True)
        loss = faceguard_loss(adv, original_embedding, use_eot=use_eot)
        identity_model.zero_grad(set_to_none=True)
        if adv.grad is not None:
            adv.grad.zero_()
        loss.backward()
        with torch.no_grad():
            adv = adv - alpha * adv.grad.sign() * mask
            adv = project_linf(adv, x, eps).detach()
    return adv


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


## 7. Baseline Methods and Robustness Transforms


In [8]:
def protect_random_noise(x, eps=MAIN_EPSILON, mask=None, **kwargs):
    if mask is None:
        mask = torch.ones_like(x[:, :1])
    return (x + torch.empty_like(x).uniform_(-eps, eps) * mask).clamp(0, 1)

def protect_fgsm(x, eps=MAIN_EPSILON, mask=None, **kwargs):
    if mask is None:
        mask = torch.ones_like(x[:, :1])
    original_embedding = identity_embedding_no_grad(x).detach()
    adv = x.detach().clone().requires_grad_(True)
    loss = F.cosine_similarity(identity_embedding_grad(adv), original_embedding).mean()
    identity_model.zero_grad(set_to_none=True)
    loss.backward()
    with torch.no_grad():
        adv = x - eps * adv.grad.sign() * mask
        return project_linf(adv, x, eps).detach()

def protect_pgd(x, eps=MAIN_EPSILON, alpha_ratio=STEP_RATIO, steps=STEPS, mask=None, **kwargs):
    if mask is None:
        mask = torch.ones_like(x[:, :1])
    original_embedding = identity_embedding_no_grad(x).detach()
    adv = (x + torch.empty_like(x).uniform_(-eps, eps) * mask).clamp(0, 1).detach()
    alpha = eps * alpha_ratio
    for _ in range(steps):
        adv.requires_grad_(True)
        loss = F.cosine_similarity(identity_embedding_grad(adv), original_embedding).mean()
        identity_model.zero_grad(set_to_none=True)
        if adv.grad is not None:
            adv.grad.zero_()
        loss.backward()
        with torch.no_grad():
            adv = adv - alpha * adv.grad.sign() * mask
            adv = project_linf(adv, x, eps).detach()
    return adv

METHODS = {
    'Random Noise': protect_random_noise,
    'FGSM': protect_fgsm,
    'PGD': protect_pgd,
    'FaceGuard': protect_faceguard,
}

def simulate_jpeg_tensor(x, quality=75):
    rgb = tensor_to_rgb_uint8(x)
    ok, encoded = cv2.imencode('.jpg', rgb_to_bgr(rgb), [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return rgb_to_tensor_device(bgr_to_rgb(decoded))

def simulate_blur_tensor(x, ksize=5):
    return rgb_to_tensor_device(cv2.GaussianBlur(tensor_to_rgb_uint8(x), (ksize, ksize), 0))

def simulate_resize_tensor(x, scale=0.80):
    rgb = tensor_to_rgb_uint8(x)
    h, w = rgb.shape[:2]
    small = cv2.resize(rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    back = cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)
    return rgb_to_tensor_device(back)


## 8. Run Cross-Dataset Evaluation
Rows include the raw result schema required by the plan.


In [9]:
def evaluate_batch(dataset_name, batch_samples, seed, method_name, method_fn, eps):
    x = torch.stack([sample['image_tensor'] for sample in batch_samples], dim=0).to(device, non_blocking=True)
    mask = make_face_mask(batch_size=x.shape[0])
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()
    protected = method_fn(x, eps=eps, alpha_ratio=STEP_RATIO, steps=STEPS, mask=mask, use_eot=USE_EOT)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    runtime_sec = (time.time() - start) / max(1, x.shape[0])

    source_rgbs = tensor_batch_to_rgb_uint8(x)
    protected_rgbs = tensor_batch_to_rgb_uint8(protected)
    clean_cosines = cosine_identity_batch(x, protected)
    jpeg = torch.cat([simulate_jpeg_tensor(protected[i:i + 1], 75) for i in range(x.shape[0])], dim=0)
    blur = torch.cat([simulate_blur_tensor(protected[i:i + 1], 5) for i in range(x.shape[0])], dim=0)
    resized = torch.cat([simulate_resize_tensor(protected[i:i + 1], 0.80) for i in range(x.shape[0])], dim=0)
    jpeg_cosines = cosine_identity_batch(x, jpeg)
    blur_cosines = cosine_identity_batch(x, blur)
    resize_cosines = cosine_identity_batch(x, resized)
    lpips_values = lpips_batch(x, protected)

    rows = []
    for i, sample in enumerate(batch_samples):
        psnr_value = psnr_np_rgb(source_rgbs[i], protected_rgbs[i])
        cos_value = float(clean_cosines[i])
        rows.append({
            'dataset': dataset_name,
            'image_id': sample['image_id'],
            'identity_id': sample['identity_id'],
            'seed': seed,
            'method': method_name,
            'epsilon': eps,
            'steps': STEPS,
            'step_ratio': STEP_RATIO,
            'cosine_similarity': cos_value,
            'psnr': psnr_value,
            'ssim': ssim_np_rgb(source_rgbs[i], protected_rgbs[i]),
            'lpips': float(lpips_values[i]),
            'success': bool(cos_value < SUCCESS_COSINE_THRESHOLD and psnr_value > SUCCESS_PSNR_THRESHOLD),
            'relaxed_success': bool(cos_value < RELAXED_SUCCESS_COSINE_THRESHOLD and psnr_value > SUCCESS_PSNR_THRESHOLD),
            'jpeg_cosine': float(jpeg_cosines[i]),
            'blur_cosine': float(blur_cosines[i]),
            'resize_cosine': float(resize_cosines[i]),
            'runtime_sec': runtime_sec,
        })
    return rows

all_rows = []
for seed in SEEDS:
    set_seed(seed)
    for dataset_name, samples in datasets.items():
        print(f'[run] seed={seed} dataset={dataset_name} N={len(samples)}')
        for eps in EPSILON_LIST:
            for method_name, method_fn in METHODS.items():
                total_batches = (len(samples) + BASELINE_BATCH_SIZE - 1) // BASELINE_BATCH_SIZE
                for _, batch_samples in tqdm(chunked(samples, BASELINE_BATCH_SIZE), total=total_batches, desc=f'{dataset_name} {method_name} eps={eps*255:.0f}/255'):
                    all_rows.extend(evaluate_batch(dataset_name, batch_samples, seed, method_name, method_fn, eps))

raw_df = pd.DataFrame(all_rows)
raw_df.head()


[run] seed=0 dataset=LFW N=500


LFW Random Noise eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

LFW FGSM eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

LFW PGD eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

LFW FaceGuard eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

...

VGGFace2 Random Noise eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

VGGFace2 FGSM eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

VGGFace2 PGD eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

VGGFace2 FaceGuard eps=12/255:   0%|          | 0/63 [00:00<?, ?it/s]

,dataset,image_id,identity_id,seed,method,epsilon,steps,step_ratio,cosine_similarity,psnr,ssim,lpips,success,relaxed_success,jpeg_cosine,blur_cosine,resize_cosine,runtime_sec
0,LFW,lfw_07108,Joe Friedberg,0,Random Noise,0.047059,35,0.125,0.991257,34.741669,0.913733,0.127375,False,False,0.989038,0.994730,0.995063,0.003506
1,LFW,lfw_03731,Christopher Amolsch,0,Random Noise,0.047059,35,0.125,0.995459,34.765319,0.906700,0.143947,False,False,0.994974,0.996303,0.997201,0.003506
2,LFW,lfw_07277,George W Bush,0,Random Noise,0.047059,35,0.125,0.993047,34.755055,0.885973,0.200661,False,False,0.992072,0.994741,0.995597,0.003506
3,LFW,lfw_02032,Na Na Keum,0,Random Noise,0.047059,35,0.125,0.985626,34.785040,0.875919,0.245479,False,False,0.976747,0.988406,0.989724,0.003506
4,LFW,lfw_02352,George W Bush,0,Random Noise,0.047059,35,0.125,0.989345,34.753726,0.910671,0.211610,False,False,0.988784,0.996482,0.994716,0.003506


## 9. Save Required CSV Outputs


In [10]:
main_df = raw_df[(raw_df['method'].eq('FaceGuard')) & (np.isclose(raw_df['epsilon'], MAIN_EPSILON))].copy()
robustness_df = main_df[['dataset', 'image_id', 'identity_id', 'seed', 'method', 'epsilon', 'steps', 'step_ratio', 'cosine_similarity', 'jpeg_cosine', 'blur_cosine', 'resize_cosine']].copy()
baseline_df = raw_df[np.isclose(raw_df['epsilon'], MAIN_EPSILON)].copy()

raw_columns = [
    'dataset', 'image_id', 'identity_id', 'seed', 'method', 'epsilon', 'steps', 'step_ratio',
    'cosine_similarity', 'psnr', 'ssim', 'lpips', 'success', 'jpeg_cosine', 'blur_cosine', 'resize_cosine'
]
save_csv(main_df[raw_columns + ['relaxed_success', 'runtime_sec']], os.path.join(RESULTS_DIR, 'results_cross_dataset_lfw_celeba_vggface2.csv'))
save_csv(robustness_df, os.path.join(RESULTS_DIR, 'results_robustness_lfw_celeba_vggface2.csv'))
save_csv(baseline_df[raw_columns + ['relaxed_success', 'runtime_sec']], os.path.join(RESULTS_DIR, 'results_baselines_lfw_celeba_vggface2.csv'))


[saved csv] /content/drive/My Drive/FaceGuard/experiments/results/results_cross_dataset_lfw_celeba_vggface2.csv
[saved csv] /content/drive/My Drive/FaceGuard/experiments/results/results_robustness_lfw_celeba_vggface2.csv
[saved csv] /content/drive/My Drive/FaceGuard/experiments/results/results_baselines_lfw_celeba_vggface2.csv


'/content/drive/My Drive/FaceGuard/experiments/results/results_baselines_lfw_celeba_vggface2.csv'

## 10. Required Tables


In [11]:
def mean_table(df, group_cols):
    return df.groupby(group_cols).agg(
        N=('image_id', 'nunique'),
        CosSim=('cosine_similarity', 'mean'),
        PSNR=('psnr', 'mean'),
        SSIM=('ssim', 'mean'),
        LPIPS=('lpips', 'mean'),
        Success_Rate=('success', 'mean'),
        Relaxed_Success_Rate=('relaxed_success', 'mean'),
    ).reset_index()

table1 = mean_table(main_df, ['dataset'])
table1 = table1[['dataset', 'N', 'CosSim', 'PSNR', 'SSIM', 'LPIPS', 'Success_Rate', 'Relaxed_Success_Rate']]
table1.columns = ['Dataset', 'N', 'CosSim', 'PSNR', 'SSIM', 'LPIPS', 'Success Rate', 'Relaxed Success Rate']

lfw_ref = table1[table1['Dataset'].eq('LFW')].iloc[0]
table2 = table1[['Dataset', 'CosSim', 'PSNR', 'Success Rate']].copy()
table2['CosSim Degradation'] = table2['CosSim'] - float(lfw_ref['CosSim'])
table2['PSNR Degradation'] = float(lfw_ref['PSNR']) - table2['PSNR']
table2['Success Rate Drop'] = float(lfw_ref['Success Rate']) - table2['Success Rate']
table2 = table2[['Dataset', 'CosSim Degradation', 'PSNR Degradation', 'Success Rate Drop']]

table3 = main_df.groupby('dataset').agg(
    N=('image_id', 'nunique'),
    Clean_CosSim=('cosine_similarity', 'mean'),
    JPEG_CosSim=('jpeg_cosine', 'mean'),
    Blur_CosSim=('blur_cosine', 'mean'),
    Resize_CosSim=('resize_cosine', 'mean'),
).reset_index()
table3.columns = ['Dataset', 'N', 'Clean CosSim', 'JPEG CosSim', 'Blur CosSim', 'Resize CosSim']

table4 = mean_table(baseline_df, ['dataset', 'method'])
table4 = table4[['dataset', 'method', 'N', 'CosSim', 'PSNR', 'SSIM', 'LPIPS', 'Success_Rate', 'Relaxed_Success_Rate']]
table4.columns = ['Dataset', 'Method', 'N', 'CosSim', 'PSNR', 'SSIM', 'LPIPS', 'Success Rate', 'Relaxed Success Rate']

display(table1)
display(table2)
display(table3)
display(table4)

save_latex(table1, os.path.join(TABLE_DIR, 'table_cross_dataset_lfw_celeba_vggface2.tex'))
save_latex(table2, os.path.join(TABLE_DIR, 'table_dataset_shift_degradation_lfw_celeba_vggface2.tex'))
save_latex(table3, os.path.join(TABLE_DIR, 'table_robustness_lfw_celeba_vggface2.tex'))
save_latex(table4, os.path.join(TABLE_DIR, 'table_baseline_comparison_lfw_celeba_vggface2.tex'))


,Dataset,N,CosSim,PSNR,SSIM,LPIPS,Success Rate,Relaxed Success Rate
0,CelebA,500,-0.922495,32.756761,0.894632,0.085319,1.0,1.0
1,LFW,500,-0.912413,33.103075,0.859612,0.291960,1.0,1.0
2,VGGFace2,388,-0.923908,32.852526,0.897639,0.091041,1.0,1.0


,Dataset,CosSim Degradation,PSNR Degradation,Success Rate Drop
0,CelebA,-0.010082,0.346314,0.0
1,LFW,0.000000,0.000000,0.0
2,VGGFace2,-0.011496,0.250549,0.0


,Dataset,N,Clean CosSim,JPEG CosSim,Blur CosSim,Resize CosSim
0,CelebA,500,-0.922495,-0.867704,-0.843626,-0.912030
1,LFW,500,-0.912413,-0.863224,-0.837214,-0.902568
2,VGGFace2,388,-0.923908,-0.871865,-0.854598,-0.915052


,Dataset,Method,N,CosSim,PSNR,SSIM,LPIPS,Success Rate,Relaxed Success Rate
0,CelebA,FGSM,500,0.866191,inf,0.947928,0.050202,0.0,0.002
1,CelebA,FaceGuard,500,-0.922495,32.756761,0.894632,0.085319,1.0,1.000
2,CelebA,PGD,500,-0.920605,33.191028,0.898282,0.086491,1.0,1.000
3,CelebA,Random Noise,500,0.993076,34.871588,0.916743,0.043436,0.0,0.000
4,LFW,FGSM,500,0.869163,inf,0.925175,0.147146,0.0,0.014
5,LFW,FaceGuard,500,-0.912413,33.103075,0.859612,0.291960,1.0,1.000
6,LFW,PGD,500,-0.908947,33.438653,0.862089,0.310847,1.0,1.000
7,LFW,Random Noise,500,0.980776,34.770434,0.880480,0.228590,0.0,0.000
8,VGGFace2,FGSM,388,0.860635,inf,0.946251,0.053969,0.0,0.006
9,VGGFace2,FaceGuard,388,-0.923908,32.852526,0.897639,0.091041,1.0,1.000


[saved table] /content/drive/My Drive/FaceGuard/experiments/tables/table_cross_dataset_lfw_celeba_vggface2.tex
[saved table] /content/drive/My Drive/FaceGuard/experiments/tables/table_dataset_shift_degradation_lfw_celeba_vggface2.tex
[saved table] /content/drive/My Drive/FaceGuard/experiments/tables/table_robustness_lfw_celeba_vggface2.tex
[saved table] /content/drive/My Drive/FaceGuard/experiments/tables/table_baseline_comparison_lfw_celeba_vggface2.tex


'/content/drive/My Drive/FaceGuard/experiments/tables/table_baseline_comparison_lfw_celeba_vggface2.tex'

## 11. Required Figures


In [12]:
def save_bar(df, x, y, path, title, ylabel, hue=None):
    plt.figure(figsize=(8, 4.8))
    if hue is None:
        plt.bar(df[x], df[y], color=['#2f6f9f', '#d85832', '#4f8a55'][:len(df)])
        plt.xticks(rotation=0)
    else:
        pivot = df.pivot(index=x, columns=hue, values=y)
        pivot.plot(kind='bar', figsize=(9, 5), ax=plt.gca())
        plt.xticks(rotation=0)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel('')
    plt.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight')
    plt.close()
    print('[saved figure]', path)

save_bar(table1, 'Dataset', 'CosSim', os.path.join(FIGURE_DIR, 'cosine_by_dataset_lfw_celeba_vggface2.png'), 'FaceGuard Cosine Similarity by Dataset', 'Cosine similarity')
save_bar(table1, 'Dataset', 'PSNR', os.path.join(FIGURE_DIR, 'psnr_by_dataset_lfw_celeba_vggface2.png'), 'FaceGuard PSNR by Dataset', 'PSNR (dB)')

robust_long = table3.melt(id_vars=['Dataset', 'N'], value_vars=['Clean CosSim', 'JPEG CosSim', 'Blur CosSim', 'Resize CosSim'], var_name='Transform', value_name='CosSim')
save_bar(robust_long, 'Dataset', 'CosSim', os.path.join(FIGURE_DIR, 'robustness_by_dataset_lfw_celeba_vggface2.png'), 'Robustness Cosine Similarity by Dataset', 'Cosine similarity', hue='Transform')

plt.figure(figsize=(7, 5))
for dataset_name, sub in main_df.groupby('dataset'):
    plt.scatter(sub['psnr'], sub['cosine_similarity'], s=18, alpha=0.45, label=dataset_name)
plt.axhline(SUCCESS_COSINE_THRESHOLD, color='black', linewidth=1, linestyle='--')
plt.axvline(SUCCESS_PSNR_THRESHOLD, color='black', linewidth=1, linestyle='--')
plt.xlabel('PSNR (dB)')
plt.ylabel('Cosine similarity')
plt.title('FaceGuard Tradeoff: PSNR vs Cosine Similarity')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
tradeoff_path = os.path.join(FIGURE_DIR, 'tradeoff_psnr_cosine_lfw_celeba_vggface2.png')
plt.savefig(tradeoff_path, dpi=220, bbox_inches='tight')
plt.close()
print('[saved figure]', tradeoff_path)


[saved figure] /content/drive/My Drive/FaceGuard/experiments/figures/cosine_by_dataset_lfw_celeba_vggface2.png
[saved figure] /content/drive/My Drive/FaceGuard/experiments/figures/psnr_by_dataset_lfw_celeba_vggface2.png
[saved figure] /content/drive/My Drive/FaceGuard/experiments/figures/robustness_by_dataset_lfw_celeba_vggface2.png
[saved figure] /content/drive/My Drive/FaceGuard/experiments/figures/tradeoff_psnr_cosine_lfw_celeba_vggface2.png


## 12. Final Claim Helper
Use the claim text only when the generated tables support it.


In [13]:
claim = """The results show that FaceGuard maintains low identity cosine similarity and high perceptual fidelity across LFW, CelebA, and VGGFace2. The small degradation in cosine similarity, PSNR, and success rate indicates that the proposed method is not strongly dependent on a specific dataset. Moreover, FaceGuard remains effective after JPEG compression, Gaussian blur, and resize/downscale transformations, suggesting that its protection generalizes across dataset shifts and common post-processing operations."""

print('Review Table 1, Table 2, and Table 3 before using this claim:')
print(claim)


Review Table 1, Table 2, and Table 3 before using this claim:
The results show that FaceGuard maintains low identity cosine similarity and high perceptual fidelity across LFW, CelebA, and VGGFace2. The small degradation in cosine similarity, PSNR, and success rate indicates that the proposed method is not strongly dependent on a specific dataset. Moreover, FaceGuard remains effective after JPEG compression, Gaussian blur, and resize/downscale transformations, suggesting that its protection generalizes across dataset shifts and common post-processing operations.
